<a href="https://colab.research.google.com/github/77marco/Challenge-TelecomX-1-DataScience/blob/main/TelecomX_LATAM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#📌 Extracción

In [1]:
import pandas as pd

# Esta es la URL "Raw" que obtuviste
url_raw = "https://raw.githubusercontent.com/ingridcristh/challenge2-data-science-LATAM/refs/heads/main/TelecomX_Data.json"

# Extraemos los datos
df_telecom = pd.read_json(url_raw)

# Mostramos los primeros datos para confirmar
df_telecom.head()

,customerID,Churn,customer,phone,internet,account
0,0002-ORFBO,No,"{'gender': 'Female', 'SeniorCitizen': 0, 'Part...","{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'DSL', 'OnlineSecurity': '...","{'Contract': 'One year', 'PaperlessBilling': '..."
1,0003-MKNFE,No,"{'gender': 'Male', 'SeniorCitizen': 0, 'Partne...","{'PhoneService': 'Yes', 'MultipleLines': 'Yes'}","{'InternetService': 'DSL', 'OnlineSecurity': '...","{'Contract': 'Month-to-month', 'PaperlessBilli..."
2,0004-TLHLJ,Yes,"{'gender': 'Male', 'SeniorCitizen': 0, 'Partne...","{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'Fiber optic', 'OnlineSecu...","{'Contract': 'Month-to-month', 'PaperlessBilli..."
3,0011-IGKFF,Yes,"{'gender': 'Male', 'SeniorCitizen': 1, 'Partne...","{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'Fiber optic', 'OnlineSecu...","{'Contract': 'Month-to-month', 'PaperlessBilli..."
4,0013-EXCHZ,Yes,"{'gender': 'Female', 'SeniorCitizen': 1, 'Part...","{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'Fiber optic', 'OnlineSecu...","{'Contract': 'Month-to-month', 'PaperlessBilli..."


#🔧 Transformación

In [2]:
import pandas as pd

# 1. "Aplanamos" los diccionarios JSON en columnas independientes
df_customer = pd.json_normalize(df_telecom['customer'])
df_phone = pd.json_normalize(df_telecom['phone'])
df_internet = pd.json_normalize(df_telecom['internet'])
df_account = pd.json_normalize(df_telecom['account'])

# 2. Unimos todo en un solo DataFrame 'limpio'
# Concatenamos customerID y Churn con las nuevas columnas creadas
df_telecom_final = pd.concat([df_telecom[['customerID', 'Churn']],
                             df_customer, df_phone, df_internet, df_account], axis=1)

# 3. Verificamos la estructura y tipos de datos
print("Resumen del nuevo conjunto de datos:")
df_telecom_final.info()

# Mostramos las primeras filas para confirmar que ya no hay llaves {}
df_telecom_final.head()

Resumen del nuevo conjunto de datos:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7267 entries, 0 to 7266
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7267 non-null   object 
 1   Churn             7267 non-null   object 
 2   gender            7267 non-null   object 
 3   SeniorCitizen     7267 non-null   int64  
 4   Partner           7267 non-null   object 
 5   Dependents        7267 non-null   object 
 6   tenure            7267 non-null   int64  
 7   PhoneService      7267 non-null   object 
 8   MultipleLines     7267 non-null   object 
 9   InternetService   7267 non-null   object 
 10  OnlineSecurity    7267 non-null   object 
 11  OnlineBackup      7267 non-null   object 
 12  DeviceProtection  7267 non-null   object 
 13  TechSupport       7267 non-null   object 
 14  StreamingTV       7267 non-null   object 
 15  StreamingMovies   7267 non-null   object 
 16  Contr

,customerID,Churn,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,...,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,Charges.Monthly,Charges.Total
0,0002-ORFBO,No,Female,0,Yes,Yes,9,Yes,No,DSL,...,Yes,No,Yes,Yes,No,One year,Yes,Mailed check,65.6,593.3
1,0003-MKNFE,No,Male,0,No,No,9,Yes,Yes,DSL,...,No,No,No,No,Yes,Month-to-month,No,Mailed check,59.9,542.4
2,0004-TLHLJ,Yes,Male,0,No,No,4,Yes,No,Fiber optic,...,No,Yes,No,No,No,Month-to-month,Yes,Electronic check,73.9,280.85
3,0011-IGKFF,Yes,Male,1,Yes,No,13,Yes,No,Fiber optic,...,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,98.0,1237.85
4,0013-EXCHZ,Yes,Female,1,Yes,No,3,Yes,No,Fiber optic,...,No,No,Yes,Yes,No,Month-to-month,Yes,Mailed check,83.9,267.4


In [3]:
# 1. Convertimos Charges.Total a numérico.
# 'errors=coerce' transformará cualquier espacio vacío en NaN (nulo) para que no falle.
df_telecom_final['Charges.Total'] = pd.to_numeric(df_telecom_final['Charges.Total'], errors='coerce')

# 2. Verificamos si aparecieron nulos tras la conversión
print(f"Nulos en Charges.Total: {df_telecom_final['Charges.Total'].isnull().sum()}")

# 3. Si hay nulos, los llenamos con 0 o con el valor de Monthly (depende de tu criterio)
df_telecom_final['Charges.Total'] = df_telecom_final['Charges.Total'].fillna(0)

# 4. Verificamos tipos finales
df_telecom_final.dtypes[['Charges.Monthly', 'Charges.Total']]

Nulos en Charges.Total: 11


,0
Charges.Monthly,float64
Charges.Total,float64


In [4]:
# 1. Verificamos si hay filas duplicadas
duplicados = df_telecom_final.duplicated().sum()
print(f"Registros duplicados: {duplicados}")

# 2. Verificamos los valores únicos en la columna objetivo (Churn)
# para asegurar que no haya errores de escritura
print(f"Valores únicos en Churn: {df_telecom_final['Churn'].unique()}")

# 3. Revisamos consistencia en una categoría clave como 'InternetService'
print(f"Categorías en InternetService: {df_telecom_final['InternetService'].unique()}")

Registros duplicados: 0
Valores únicos en Churn: ['No' 'Yes' '']
Categorías en InternetService: ['DSL' 'Fiber optic' 'No']


In [5]:
# 1. Contamos cuántos registros tienen el campo Churn vacío
vacios_churn = (df_telecom_final['Churn'] == '').sum()
print(f"Registros con Churn vacío: {vacios_churn}")

# 2. Nos quedamos solo con los registros que SÍ tienen 'Yes' o 'No'
df_telecom_final = df_telecom_final[df_telecom_final['Churn'] != '']

# 3. Verificamos que ya no existan
print(f"Nuevos valores únicos en Churn: {df_telecom_final['Churn'].unique()}")
print(f"Tamaño final del dataset: {df_telecom_final.shape}")

Registros con Churn vacío: 224
Nuevos valores únicos en Churn: ['No' 'Yes']
Tamaño final del dataset: (7043, 21)


#📊 Carga y análisis

#📄Informe final